# LDA MODEL WITH SKLEARN
- References:
1. https://medium.com/data-science/practical-guide-to-topic-modeling-with-lda-05cd6b027bdf
2. https://machinelearningplus.com/nlp/topic-modeling-python-sklearn-examples/#1introduction

In [2]:
import pandas as pd
import spacy
import numpy as np 
from spacy.language import Language
from spacy.schemas import ConfigSchemaNlp
ConfigSchemaNlp.model_rebuild()
nlp = spacy.load('en_core_web_lg')
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import RandomizedSearchCV
from sklearn.datasets import fetch_20newsgroups
from scipy.stats import loguniform, randint, uniform

# Plotting tools
import pyLDAvis
import pyLDAvis.lda_model
import matplotlib.pyplot as plt
%matplotlib inline
# own function 
import lda_preprocess



# Preprocessing

In [4]:
df = pd.read_csv('../datasets/final/mda_shared_preprocessed.csv')
docs = (
    df.groupby(['doc_id'])['sentence'].apply(' '.join)
      .reset_index()
      .rename(columns={'sentence': 'clean_text'})
)

docs.head()
# dtm, vectors, vectorizer = lda_preprocess.docs2both(docs)

,doc_id,clean_text
0,1-800-PetMeds_10-K_2020-05-26,approximately NUM of all sales were generated ...
1,1-800-PetMeds_10-K_2021-05-25,fiscal NUM started out with greater than expec...
2,1-800-PetMeds_10-K_2022-05-24,we have continued with working from home where...
3,1-800-PetMeds_10-K_2023-05-23,management does not believe that the resolutio...
4,1-800-PetMeds_10-K_2024-06-14,NUM we have direct buying relationships with a...


In [15]:
dtm.shape

(17560, 10000)

## Search for best params

### Priors setting
- default number of topics setting = 20 

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import cross_val_score
import numpy as np

# 3 principled prior combinations to test
prior_candidates = [
    {
        "name": "sparse (focused topics)",
        "doc_topic_prior": 0.1,      # low α → docs load on few topics
        "topic_word_prior": 0.01,    # low β → topics are narrow/specific
    },
    {
        "name": "symmetric (balanced)",
        "doc_topic_prior": 1/20,     # α = 1/K, common default heuristic
        "topic_word_prior": 1/20,    # β = 1/K
    },
    {
        "name": "diffuse (mixed topics)",
        "doc_topic_prior": 0.5,      # high α → docs spread across many topics
        "topic_word_prior": 0.1,     # moderate β
    },
]

n_components = 20  # fix this first, tune separately if needed
results = []

for prior in prior_candidates:
    print(f"\nFitting: {prior['name']}...")
    lda = LatentDirichletAllocation(
        n_components=n_components,
        doc_topic_prior=prior["doc_topic_prior"],
        topic_word_prior=prior["topic_word_prior"],
        learning_method="online",
        learning_decay=0.7,          # fix decay — mid-range is safest
        max_iter=20,
        random_state=42,
        n_jobs=-1
    )
    
    # single train/val split instead of cv=3 — 3x faster
    split = int(0.8 * dtm.shape[0])
    lda.fit(dtm[:split])
    score = lda.score(dtm[split:])   # per-token log-likelihood on held-out set
    
    results.append({"prior": prior["name"], "score": score, "params": prior})
    print(f"  held-out log-likelihood: {score:.2f}")

best = max(results, key=lambda x: x["score"])
print(f"\nBest prior: {best['prior']} — score: {best['score']:.2f}")

### Number of topics
### 2 phase approach: 
- coarse : 10 to 100 topics, interval of 10
- fine: 50-70 (identified from coarse), intercal of 5

### Phase 1: Coarse Search

In [ ]:
# Phase 1: Coarse search — only 10 fits, low max_iter just to find elbow region
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np
import matplotlib.pyplot as plt

topic_range_coarse = range(10, 110, 10)  # 10, 20, 30, ..., 100
perplexities_coarse = []

for n_topics in topic_range_coarse:
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        doc_topic_prior=0.5,       
        topic_word_prior=0.1, 
        learning_method='online',
        batch_size=512,
        max_iter=5,             # intentionally low — just finding the elbow region
        random_state=42,
        n_jobs=-1
    )
    lda.fit(dtm)
    perplexities_coarse.append(lda.perplexity(dtm))
    print(f"n_topics={n_topics:3d} | Perplexity: {perplexities_coarse[-1]:.2f}")


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(list(topic_range_coarse), perplexities_coarse, marker='o', linewidth=2, color='tomato')
plt.axvspan(50, 70, alpha=0.2, color='blue', label='Elbow region (50–70)') 
plt.title("Phase 1: Coarse Perplexity Search")
plt.xlabel("Number of Topics")
plt.ylabel("Perplexity (lower = better)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Phase 2: Fine search 

In [ ]:
# Phase 2: Fine search — zoom into elbow region
topic_range_fine = range(50,70,1)
perplexities_fine = []

for n_topics in topic_range_fine:
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        doc_topic_prior=0.5,       
        topic_word_prior=0.1, 
        learning_method='online',
        batch_size=512,
        max_iter=10,     
        random_state=42,
        n_jobs=-1
    )
    lda.fit(dtm)
    perplexities_fine.append(lda.perplexity(dtm))
    print(f"n_topics={n_topics:3d} | Perplexity: {perplexities_fine[-1]:.2f}")

In [ ]:

# --- Plot Phase 2 ---
plt.figure(figsize=(10, 5))
plt.plot(list(topic_range_fine), perplexities_fine, marker='o', linewidth=2, color='tomato')
plt.axvspan(59.9, 60.1, alpha=0.2, color='blue', label='Best N') 
plt.title("Phase 2: Fine Perplexity Search")
plt.xlabel("Number of Topics")
plt.ylabel("Perplexity (lower = better)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_n = list(topic_range_fine)[np.argmin(perplexities_fine)]
print(f"\nOptimal number of topics: {best_n}")

## Best Model
### Using the settings found earlier
- priors: "doc_topic_prior": 0.5,  "topic_word_prior": 0.1,  
- number of topics: 60


In [ ]:
sklearn_lda_model = LatentDirichletAllocation(
    n_components=60,
    learning_method="online",
    doc_topic_prior=0.5,       
    topic_word_prior=0.1, 
    total_samples=len(docs),
    batch_size=2000,
    learning_decay=0.7,
    learning_offset=100,
    random_state=42,
)
sklearn_lda_model.fit(dtm)


In [ ]:
docs.head()

# Inspect Topics and Find Key words through TF-IDF

- use tf-idf to bring rare words to the top 

In [ ]:
def get_distinctive_words(lda_model, vectorizer, n_words=50):
    """Reweight topic words by how unique they are across topics"""
    components = lda_model.components_
    
    # Normalize each topic to a probability distribution
    topic_word_prob = components / components.sum(axis=1, keepdims=True)
    
    # IDF: penalize words that are common across all topics
    topic_freq = (topic_word_prob > 0.001).sum(axis=0)  # how many topics use this word
    idf = np.log(components.shape[0] / (topic_freq + 1))
    
    # Reweight
    distinctive = topic_word_prob * idf
    
    words = vectorizer.get_feature_names_out()
    for i, topic in enumerate(distinctive):
        top_words = [words[j] for j in topic.argsort()[-n_words:][::-1]]
        print(f"Topic {i:02d}: {', '.join(top_words)}")

get_distinctive_words(sklearn_lda_model, vectorizer)



# Label topics

# Find topics in new doc

In [ ]:
def predict_topic(text, nlp=nlp):
    # preprocess txt 
    docs = lda_preprocess.preprocess_docs(text)
    
    #vectorise
    dtm, _, vectorizer = lda_preprocess.docs2both(docs)

    # LDA Transform
    topic_probability_scores = sklearn_lda_model.transform(dtm)
    topic = df_topic_keywords.iloc[np.argmax(topic_probability_scores), :].values.tolist()
    return topic, topic_probability_scores

# Predict the topic
mytext = ["insert a 2026 10 report"]
topic, prob_scores = predict_topic(text = mytext)
print(topic)